# Uncensored Coder — Colab Notebook

This notebook runs an existing open model in your own Google Colab:

**Model:** `huihui-ai/Qwen2.5-Coder-7B-Instruct-abliterated`
- A coder-specialized model (Qwen2.5-Coder-7B-Instruct) with the refusal direction removed ("abliterated" = uncensored).
- License: Apache-2.0. Runs here in 4-bit, so it fits a free Colab T4 GPU (16 GB).

Note: this is a ready-made open model, not the from-scratch model being built in the Bangla Companion project.

## How to use
1. Set the runtime to GPU: **Runtime → Change runtime type → T4 GPU**.
2. Run the cells from top to bottom (Shift+Enter).
3. In the last cell, type your coding question and press Enter. Write in Bengali or English — the model answers in the language you use.
4. In the chat: type `/clean` to start a fresh conversation, `/exit` to stop the chat loop (then just rerun the cell to chat again).

In [ ]:
# Step 1 — Check the GPU
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("No GPU found. Go to Runtime -> Change runtime type -> T4 GPU, then rerun this cell.")

In [ ]:
# Step 2 — Install the libraries (pinned versions, quiet)
%pip install -q transformers==4.49.0 accelerate==1.4.0 bitsandbytes==0.45.3
print("Install done.")

In [ ]:
# Step 3 — Load the model in 4-bit (fits a T4: roughly 5 GB of VRAM)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "huihui-ai/Qwen2.5-Coder-7B-Instruct-abliterated"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print("Model loaded:", MODEL_ID)

In [ ]:
# Step 4 — Chat helper
SYSTEM_PROMPT = (
    "You are an expert coding assistant. Write clean, working code and explain it briefly. "
    "Answer in the same language the user writes in: if the user writes Bengali, answer in Bengali; "
    "if the user writes English, answer in English. Code and code comments stay in English."
)

history = [{"role": "system", "content": SYSTEM_PROMPT}]

def chat(prompt, max_new_tokens=2048):
    """Send one message, keep the conversation history, return the reply."""
    history.append({"role": "user", "content": prompt})
    text = tokenizer.apply_chat_template(history, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    history.append({"role": "assistant", "content": reply})
    return reply

# Quick test (you can delete this line):
print(chat("Write a Python function that reverses a string."))

In [ ]:
# Step 5 — Interactive chat loop. Rerun this cell any time to keep chatting.
# Type /clean for a fresh conversation, /exit to stop.
while True:
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nChat stopped.")
        break
    if not user_input:
        continue
    if user_input.lower() == "/exit":
        print("Chat stopped. Rerun this cell to chat again.")
        break
    if user_input.lower() == "/clean":
        history[:] = [{"role": "system", "content": SYSTEM_PROMPT}]
        print("New conversation started.")
        continue
    print("\nCoder:\n" + chat(user_input) + "\n")